In [ ]:
import numpy as np
import json
from datasets import load_dataset
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
from transformers import AutoTokenizer, AutoModel
import torch
from tqdm import tqdm
from typing import List


class GENALMEmbeddingExtractor:
    """
    Класс извлечения эмбеддингов с помощью GENA-LM.
    """
    def __init__(self,
                 model_name: str = 'AIRI-Institute/gena-lm-bert-base',
                 device: str = None):
        self.device = device or ('cuda' if torch.cuda.is_available() else 'cpu')
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModel.from_pretrained(model_name, trust_remote_code=True)
        self.model.to(self.device)
        self.model.eval()

    def extract_embeddings(self,
                           sequences: List[str],
                           batch_size: int = 8,
                           pooling: str = 'mean') -> np.ndarray:
        """
        Извлечение эмбеддингов для получения DNA/RNA-последовательностей.

        Args:
            sequences: list of str sequences
            batch_size: number of sequences per batch
            pooling: 'mean' for average pooling, 'cls' for CLS token

        Returns:
            np.ndarray of shape (len(sequences), hidden_size)
        """
        all_embs = []
        for i in range(0, len(sequences), batch_size):
            batch_seqs = sequences[i:i + batch_size]
            encoded = self.tokenizer(batch_seqs,
                                     return_tensors='pt',
                                     padding=True,
                                     truncation=True)
            input_ids = encoded['input_ids'].to(self.device)
            attention_mask = encoded['attention_mask'].to(self.device)

            with torch.no_grad():
                outputs = self.model.bert(input_ids=input_ids,
                                     attention_mask=attention_mask)
                last_hidden = outputs.last_hidden_state  # (B, L, H)

                if pooling == 'cls':
                    # Use the [CLS] token representation
                    emb = last_hidden[:, 0, :].cpu().numpy()
                else:
                    # Mean-pooling over the token dimension
                    mask = attention_mask.unsqueeze(-1)
                    sum_emb = (last_hidden * mask).sum(1)
                    lengths = mask.sum(1)
                    emb = (sum_emb / lengths).cpu().numpy()

            all_embs.append(emb)

        return np.vstack(all_embs)



ds = load_dataset("InstaDeepAI/nucleotide_transformer_downstream_tasks")
train_ds, test_ds = ds['train'], ds['test']

extractor = GENALMEmbeddingExtractor()

PARAMS_LOGREG = {'max_iter': 1000, 'random_state': 42}
BATCH_SIZE = 16
PATH_TO_SAVE_OUTPUT = '/kaggle/working'

# Baseline: обучение на полном наборе
baseline = {}
for task in tqdm(set(train_ds['task']), desc='Baseline'):
    tr = train_ds.filter(lambda x, t=task: x['task'] == t)
    te = test_ds.filter(lambda x, t=task: x['task'] == t)
    seqs_tr, y_tr = tr['sequence'], np.array(tr['label'])
    seqs_te, y_te = te['sequence'], np.array(te['label'])

    X_tr = extractor.extract_embeddings(seqs_tr, batch_size=BATCH_SIZE)
    X_te = extractor.extract_embeddings(seqs_te, batch_size=BATCH_SIZE)

    clf = LogisticRegression(**PARAMS_LOGREG)
    Xf = X_tr.reshape(-1, 1) if X_tr.ndim == 1 or X_tr.shape[1] == 1 else X_tr
    Xt = X_te.reshape(-1, 1) if X_te.ndim == 1 or X_te.shape[1] == 1 else X_te
    clf.fit(Xf, y_tr)
    preds = clf.predict(Xt)

    baseline[task] = {
        'accuracy': float(accuracy_score(y_te, preds)),
        'f1_score': float(f1_score(y_te, preds, average='macro'))
    }
    with open(f'{PATH_TO_SAVE_OUTPUT}/results_genalm_task-{task}_baseline.json', 'w') as f:
        json.dump(baseline, f, indent=4)

# Few-shot эксперименты
def few_shot(train, test, ks=(1, 5, 10, 20), trials=5):
    res = {}
    rng = np.random.RandomState(42)
    for task in tqdm(set(train['task']), desc='Few-shot'):
        tr = train.filter(lambda x, t=task: x['task'] == t)
        te = test.filter(lambda x, t=task: x['task'] == t)
        seqs_tr, y_tr = tr['sequence'], np.array(tr['label'])
        seqs_te, y_te = te['sequence'], np.array(te['label'])
        X_te = extractor.extract_embeddings(seqs_te, batch_size=BATCH_SIZE)
        res[task] = {}

        for k in ks:
            accs, f1s = [], []
            for _ in range(trials):
                idxs = []
                for lbl in np.unique(y_tr):
                    locs = np.where(y_tr == lbl)[0]
                    choice = rng.choice(locs, size=min(k, len(locs)), replace=False)
                    idxs.extend(choice.tolist())
                X_k = extractor.extract_embeddings([seqs_tr[i] for i in idxs], batch_size=BATCH_SIZE)
                y_k = y_tr[idxs]
                clf = LogisticRegression(**PARAMS_LOGREG)
                Xf = X_k.reshape(-1, 1) if X_k.ndim == 1 or X_k.shape[1] == 1 else X_k
                Xt = X_te.reshape(-1, 1) if X_te.ndim == 1 or X_te.shape[1] == 1 else X_te
                clf.fit(Xf, y_k)
                p = clf.predict(Xt)
                accs.append(accuracy_score(y_te, p))
                f1s.append(f1_score(y_te, p, average='macro'))
            res[task][k] = {'accuracy': float(np.mean(accs)), 'f1_score': float(np.mean(f1s))}
            with open(f'{PATH_TO_SAVE_OUTPUT}/results_genalm_task-{task}_k-{k}.json', 'w') as f:
                json.dump(res, f, indent=4)

    return res

results_kshot = few_shot(train_ds, test_ds)

output = {'full': baseline, 'kshot': results_kshot, 'params': PARAMS_LOGREG}
with open(f'{PATH_TO_SAVE_OUTPUT}/results_genalm.json', 'w') as f:
    json.dump(output, f, indent=4)


Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

Baseline:   0%|          | 0/18 [00:00<?, ?it/s]Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.
/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_logistic.py:458: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
Baseline:   6%|▌         | 1/18 [02:00<34:00, 120.01s/it]

Filter:   0%|          | 0/461850 [00:00<?, ? examples/s]

Filter:   0%|          | 0/48797 [00:00<?, ? examples/s]

/usr/local/lib/python3.11/dist-packages/transformers/modeling_utils.py:1614: FutureWarning: The `device` argument is deprecated and will be removed in v5 of Transformers.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_logistic.py:458: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
Baseline:  11%|█         | 2/18 [02:24<17:05, 64.10s/it] 

Filter:   0%|          | 0/461850 [00:00<?, ? examples/s]

Filter:   0%|          | 0/48797 [00:00<?, ? examples/s]

/usr/local/lib/python3.11/dist-packages/transformers/modeling_utils.py:1614: FutureWarning: The `device` argument is deprecated and will be removed in v5 of Transformers.
  warnings.warn(
